# `midas-stress` — tutorial

This notebook is a self-contained, runnable tour of every public capability of the `midas-stress` package:

1. Environment setup and loading data from any source
2. Just-compute-stress — the 5-line path
3. Using a known applied macroscopic stress (equilibrium with load)
4. **d0 unknown** — recovering the strain-free lattice from mechanical equilibrium
5. Stress resolved along directions, planes, principal axes, and in other frames
6. **CRSS / Schmid / resolved shear** — publication-ready slip-system analysis
7. Using real MIDAS output (`Grains.csv` / `GrainsSim.csv`)
8. Reference tables (function cheat-sheet, units, frame conventions)
9. Decision tree — pick the right function for your case
10. Intentional non-goals

Every section runs independently. Figures are saved as PNG alongside this notebook because matplotlib inline rendering can be flaky — the path is printed after every plot.

**Audience**: anyone with per-grain orientations and strains (from MIDAS, hexrd, ImageD11, DAXM, or their own script). The package takes numpy arrays as inputs — no MIDAS dependency.

## 1. Setup

In [1]:
import os
import numpy as np
import matplotlib.pyplot as plt

import midas_stress as ms

plt.rcParams.update({
    'font.size': 12,
    'figure.figsize': (7, 4),
    'figure.dpi': 100,
    'axes.grid': True,
    'grid.alpha': 0.3,
})

SAVE_DIR = os.path.abspath('.')  # where figures are written
RNG = np.random.default_rng(0)

def save_fig(fig, name):
    path = os.path.join(SAVE_DIR, f'{name}.png')
    fig.savefig(path, bbox_inches='tight', dpi=120)
    print(f'saved: {path}')

def random_rotations(n, seed):
    """Uniform random rotation matrices via unit quaternions (no scipy dependency)."""
    rng = np.random.default_rng(seed)
    q = rng.normal(size=(n, 4))
    q /= np.linalg.norm(q, axis=1, keepdims=True)
    w, x, y, z = q[:, 0], q[:, 1], q[:, 2], q[:, 3]
    R = np.empty((n, 3, 3))
    R[:, 0, 0] = 1 - 2 * (y * y + z * z)
    R[:, 0, 1] = 2 * (x * y - w * z)
    R[:, 0, 2] = 2 * (x * z + w * y)
    R[:, 1, 0] = 2 * (x * y + w * z)
    R[:, 1, 1] = 1 - 2 * (x * x + z * z)
    R[:, 1, 2] = 2 * (y * z - w * x)
    R[:, 2, 0] = 2 * (x * z - w * y)
    R[:, 2, 1] = 2 * (y * z + w * x)
    R[:, 2, 2] = 1 - 2 * (x * x + y * y)
    return R

print('midas_stress version :', ms.__version__)
print('built-in materials   :', ms.list_materials())
print('slip families        :', ms.list_slip_families())


midas_stress version : 0.2.1
built-in materials   : ['Al', 'Au', 'CeO2', 'Cu', 'Fe', 'Ni', 'Si', 'Ti', 'W']
slip families        : ['bcc', 'bcc_110', 'bcc_112', 'bcc_123', 'bcc_all', 'fcc', 'fcc_octahedral', 'hcp', 'hcp_basal', 'hcp_prismatic', 'hcp_pyramidal_a', 'hcp_pyramidal_ca']


### 1a. Data source

Use the bundled `GrainsSim.csv` (250 grains, FCC, strain-free sim). `ms.example_data_path()` resolves to whatever ships with the install (works in both editable and wheel installs).

**Point to your own data** by replacing the path below with any MIDAS CSV — the header-driven parser handles both classic `Grains.csv` and the newer `GrainsSim.csv` / `Grains_allatonce.csv` layouts automatically.

In [2]:
# >>> change this to your own file if you like <<<
GRAINS_CSV = ms.example_data_path('GrainsSim.csv')
# GRAINS_CSV = '/path/to/your/Grains.csv'

grains = ms.read_grains(GRAINS_CSV)

print(f'file        : {GRAINS_CSV}')
print(f'N grains    : {grains["orientations"].shape[0]}')
print(f'keys        : {sorted(k for k in grains if k != "raw")}')
print()
print('strain key  : default is `grains["strain"]` = d-spacing (strain-gauge) form — historically "Kenesei" in MIDAS')
print('alternate   : `grains["strain_lattice"]` = lattice-parameter form — historically "Fable-Beaudoin" in MIDAS')

file        : /Users/hsharma/opt/MIDAS/packages/midas_stress/midas_stress/_data/GrainsSim.csv
N grains    : 250
keys        : ['columns', 'confidences', 'euler_angles', 'grain_ids', 'lattice_params', 'orientations', 'phase', 'positions', 'radii', 'rms_error', 'strain', 'strain_lattice']

strain key  : default is `grains["strain"]` = d-spacing (strain-gauge) form — historically "Kenesei" in MIDAS
alternate   : `grains["strain_lattice"]` = lattice-parameter form — historically "Fable-Beaudoin" in MIDAS


### 1b. Synthetic strain (so every example has something non-trivial to show)

`GrainsSim.csv` has zero strain by construction (it is a geometric simulation). To demonstrate the pipeline we inject a physically sensible strain distribution: a macroscopic uniaxial strain plus small grain-to-grain scatter. This mirrors what a real far-field HEDM dataset looks like after loading.

In [3]:
N = grains['orientations'].shape[0]
orientations = grains['orientations']
volumes = (4 / 3) * np.pi * grains['radii']**3
confidences = grains['confidences']
positions = grains['positions']

# Macroscopic strain (uniaxial along z, with Poisson contraction).
# eps_zz = +1e-3, Poisson nu ~ 0.3 for Cu
eps_macro = np.diag([-3e-4, -3e-4, 1e-3])

# Per-grain deviation (random symmetric ~2e-4) to simulate intergranular variation.
eps_scatter = 2e-4 * RNG.normal(size=(N, 3, 3))
eps_scatter = 0.5 * (eps_scatter + eps_scatter.swapaxes(-1, -2))

strains = eps_macro[None, :, :] + eps_scatter

# Also build a "corrupted" strain that simulates an unknown d0 error
# (isotropic strain offset of +300 ppm).
strains_with_d0_error = strains + 3e-4 * np.eye(3)[None, :, :]

print(f'synthetic strain, mean diag = {np.diag(strains.mean(axis=0))}')

synthetic strain, mean diag = [-0.00032191 -0.00029892  0.00099564]


## 2. Just compute stresses — the 5-line path

**Goal**: user has orientations, strains, volumes. Produce per-grain stress tensors.

In [4]:
# One-liner: free-standing sample (no applied load), automatic equilibrium correction.
result = ms.compute_stress(
    strain=strains,
    stiffness=ms.get_stiffness('Cu'),
    orient=orientations,
    volumes=volumes,
)

stress_raw = result['stress_raw']           # (N, 3, 3) before correction
stress_corr = result['stress_corrected']    # (N, 3, 3) after correction
von_mises = result['von_mises']             # (N,)
hydro = result['hydrostatic_corrected']     # (N,)

print(f'mean von Mises       : {von_mises.mean():.2f} GPa')
print(f'hydrostatic shift    : {result["hydrostatic_shift"]:.3e} GPa '
      f'(correction applied)')
print(f'uncertainty SE (hydro): {result["uncertainty"]["hydrostatic_se_MPa"]:.2f} MPa')

mean von Mises       : 0.09 GPa
hydrostatic shift    : -5.138e-02 GPa (correction applied)
uncertainty SE (hydro): 0.00 MPa


### 2a. Without the equilibrium correction

If you trust `d0` and don't want the correction, call `hooke_stress` directly. Inputs assumed in the **lab frame** by default — pass `frame="grain"` if your strain is in the crystal frame.

In [5]:
stress_no_correction = ms.hooke_stress(
    strains, ms.get_stiffness('Cu'),
    orient=orientations, frame='lab',
)

# Sanity: stress_raw from compute_stress matches hooke_stress (both are pre-correction).
np.testing.assert_allclose(stress_no_correction, stress_raw, atol=1e-10)
print('hooke_stress == compute_stress["stress_raw"]   ✓')

hooke_stress == compute_stress["stress_raw"]   ✓


### 2b. Inspect outputs

In [6]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

axes[0].hist(von_mises * 1e3, bins=30, color='steelblue', edgecolor='k')  # GPa -> MPa
axes[0].set_xlabel('von Mises [MPa]')
axes[0].set_ylabel('grain count')
axes[0].set_title('von Mises distribution')

axes[1].hist(ms.hydrostatic(stress_raw) * 1e3, bins=30, alpha=0.6, label='raw', color='tomato')
axes[1].hist(hydro * 1e3, bins=30, alpha=0.6, label='corrected', color='seagreen')
axes[1].set_xlabel('hydrostatic stress [MPa]')
axes[1].legend()
axes[1].set_title('equilibrium correction shifts hydrostatic only')

comp_labels = ['$\\sigma_{xx}$', '$\\sigma_{yy}$', '$\\sigma_{zz}$']
mean_comp = np.array([stress_corr[:, i, i].mean() for i in range(3)]) * 1e3
axes[2].bar(comp_labels, mean_comp, color=['c', 'c', 'C1'])
axes[2].set_ylabel('mean stress [MPa]')
axes[2].set_title('polycrystal-mean principal stress')

fig.tight_layout()
save_fig(fig, 'fig_02_compute_stress')
plt.close(fig)

saved: /Users/hsharma/opt/MIDAS/packages/midas_stress/examples/fig_02_compute_stress.png


**Gotchas** (read once, bookmark):

- `volumes` only needs to be *proportional* to grain volume. `r^3`, voxel counts, or `np.ones(N)` (equal weights) all work.
- `orient` is crystal→lab. If your code returns lab→crystal, transpose.
- Input `strain` frame must match the output frame (`"lab"` or `"grain"`).
- Stress comes out in the same units as stiffness (GPa by default). Multiply by `1e3` for MPa.

## 3. Known applied macroscopic stress

Pass a 3×3 `applied_stress` tensor. Works for any loading mode (uniaxial, biaxial, shear, arbitrary).

In [7]:
# Uniaxial 150 MPa along z, expressed in GPa
sigma_app_uni = np.diag([0.0, 0.0, 0.150])

# Biaxial 100 MPa along x and y
sigma_app_bi = np.diag([0.100, 0.100, 0.0])

# Pure shear tau=80 MPa (xy plane)
sigma_app_shear = np.array([[0.0, 0.080, 0.0],
                            [0.080, 0.0, 0.0],
                            [0.0, 0.0, 0.0]])

cases = {
    'free-standing': None,
    'uniaxial (z)':  sigma_app_uni,
    'biaxial (x, y)': sigma_app_bi,
    'pure shear':     sigma_app_shear,
}

print(f'{"load case":<18}   hydro<\u03c3>_poly [MPa]   von Mises mean [MPa]   SE [MPa]')
for label, sig_app in cases.items():
    r = ms.compute_stress(
        strain=strains, stiffness=ms.get_stiffness('Cu'),
        orient=orientations, volumes=volumes,
        applied_stress=sig_app,
    )
    h = r['hydrostatic_corrected'].mean() * 1e3
    vm = r['von_mises'].mean() * 1e3
    se = r['uncertainty']['hydrostatic_se_MPa']
    print(f'{label:<18}   {h:>14.2f}   {vm:>18.2f}   {se:>8.2f}')

load case            hydro<σ>_poly [MPa]   von Mises mean [MPa]   SE [MPa]
free-standing                 -0.00                89.66       0.00
uniaxial (z)                  50.00               171.45       0.00
biaxial (x, y)                66.67               132.37       0.00
pure shear                    -0.00               163.53       0.00


### 3a. Diagnostic: what the correction actually did

`result["uncertainty"]` reports per-component standard error of the volume-averaged stress. Rule of thumb: if `SE` is comparable to the correction magnitude, trust the **raw** stresses more than the corrected ones.

In [8]:
r = ms.compute_stress(
    strain=strains, stiffness=ms.get_stiffness('Cu'),
    orient=orientations, volumes=volumes,
    applied_stress=sigma_app_uni,
)

se = r['uncertainty']['stress_se_voigt_MPa']
mean = r['uncertainty']['stress_mean_voigt_MPa'] * 1e3  # GPa → MPa
labels = ['xx', 'yy', 'zz', 'xy', 'xz', 'yz']

fig, ax = plt.subplots()
ax.errorbar(labels, mean, yerr=se, fmt='o', capsize=5)
ax.axhline(0, color='k', lw=0.5)
ax.set_ylabel('volume-avg stress [MPa]')
ax.set_title(f'polycrystal-mean stress ± SE (applied = uniaxial 150 MPa z)')
fig.tight_layout()
save_fig(fig, 'fig_03_applied_stress_diagnostic')
plt.close(fig)

print(f'\nEffective sample size: {r["uncertainty"]["effective_n"]:.1f} (of {N} grains)')

saved: /Users/hsharma/opt/MIDAS/packages/midas_stress/examples/fig_03_applied_stress_diagnostic.png

Effective sample size: 250.0 (of 250 grains)


## 4. d0 unknown — headline feature

Every HEDM experiment has an unknown strain-free lattice parameter `d0`. A tiny error in `d0` produces a huge, **identical-for-every-grain** hydrostatic-stress artefact. `midas-stress` is the only library that fixes this directly via mechanical equilibrium.

### 4a. Why d0 matters — sensitivity

In [9]:
table = ms.d0_sensitivity_table()
mats = list(table.keys())
vals = np.array([table[m]['sensitivity_MPa_per_100ppm'] for m in mats])
pure_hydro = np.array([table[m]['is_pure_hydrostatic'] for m in mats])

fig, ax = plt.subplots()
colors = ['steelblue' if p else 'tomato' for p in pure_hydro]
ax.bar(mats, vals, color=colors)
ax.set_ylabel('stress error per 100 ppm d0 error [MPa]')
ax.set_title('d0 sensitivity (blue = cubic / pure hydrostatic; red = non-cubic)')
fig.tight_layout()
save_fig(fig, 'fig_04a_d0_sensitivity')
plt.close(fig)

print('For Ti (hexagonal) a d0 error produces BOTH hydrostatic AND')
print('deviatoric stress artefacts — handled correctly by `correct_d0`.')

saved: /Users/hsharma/opt/MIDAS/packages/midas_stress/examples/fig_04a_d0_sensitivity.png
For Ti (hexagonal) a d0 error produces BOTH hydrostatic AND
deviatoric stress artefacts — handled correctly by `correct_d0`.


### 4b. Corrupted strains — visualise the artefact

In [10]:
truth = ms.compute_stress(strain=strains, stiffness=ms.get_stiffness('Cu'),
                          orient=orientations, volumes=volumes)
bad   = ms.compute_stress(strain=strains_with_d0_error, stiffness=ms.get_stiffness('Cu'),
                          orient=orientations, volumes=volumes)

fig, ax = plt.subplots()
ax.hist(ms.hydrostatic(truth['stress_raw']) * 1e3, bins=30, alpha=0.6, label='truth (no d0 err)')
ax.hist(ms.hydrostatic(bad['stress_raw']) * 1e3, bins=30, alpha=0.6, label='300 ppm d0 err')
ax.set_xlabel('raw hydrostatic stress [MPa]')
ax.set_title('d0 error shifts EVERY grain by the same amount')
ax.legend()
fig.tight_layout()
save_fig(fig, 'fig_04b_d0_artefact')
plt.close(fig)

saved: /Users/hsharma/opt/MIDAS/packages/midas_stress/examples/fig_04b_d0_artefact.png


### 4c. Two-step equilibrium correction (`correct_d0`)

In [11]:
res = ms.correct_d0(
    strains=strains_with_d0_error,
    stiffness=ms.get_stiffness('Cu'),
    orientations=orientations,
    volumes=volumes,
    applied_stress=None,   # free-standing
)

print(f'recovered eps_iso         : {res["eps_iso"]:.3e}  (expected ≈ 3e-4)')
print(f'residual before correction: {res["residual_norm_before"] * 1e3:.1f} MPa')
print(f'residual after step 1     : {res["residual_norm_after"] * 1e3:.2e} MPa')
print(f'residual after step 2     : {res["residual_norm_2step"] * 1e3:.2e} MPa')

recovered eps_iso         : 4.249e-04  (expected ≈ 3e-4)
residual before correction: 324.6 MPa
residual after step 1     : 1.17e+02 MPa
residual after step 2     : 4.79e-14 MPa


### 4d. Recovering the true strain-free lattice parameter

`recover_d0` works straight from per-grain lattice parameters (what a peak-fitting code actually reports). You hand it an **assumed** reference that you think is correct; it returns the reference consistent with equilibrium.

In [12]:
# True Cu lattice (from GrainsSim header): 4.080 A
a0_true = 4.080
# Pretend the user supplied a wrong reference a0 = 4.082 (δa/a = +490 ppm)
a0_assumed = 4.082

# Synthesise per-grain lattice parameters consistent with a small strain + the d0 error
# (just a demo; with real data use grains['lattice_params']).
lat_true = np.array([a0_true, a0_true, a0_true, 90.0, 90.0, 90.0])
# Apply the macroscopic strain and scatter to per-grain lattice a,b,c
lat_params = np.tile(lat_true, (N, 1))
for i in range(N):
    # Infinitesimal strain -> fractional change in lattice length is the diagonal
    eps_diag = np.diag(strains[i])
    lat_params[i, 0] = a0_true * (1.0 + eps_diag[0])
    lat_params[i, 1] = a0_true * (1.0 + eps_diag[1])
    lat_params[i, 2] = a0_true * (1.0 + eps_diag[2])

assumed_ref = np.array([a0_assumed, a0_assumed, a0_assumed, 90.0, 90.0, 90.0])

rec = ms.recover_d0(
    lattice_params=lat_params,
    assumed_reference=assumed_ref,
    stiffness=ms.get_stiffness('Cu'),
    orientations=orientations,
    volumes=volumes,
)

print(f'assumed a0  : {rec["reference_assumed"][0]:.6f} A')
print(f'recovered a0: {rec["reference_recovered"][0]:.6f} A')
print(f'true a0     : {a0_true:.6f} A')
print(f'scale factor: {rec["scale_factor"]:.6f}')
print(f'eps_iso fit : {rec["eps_iso"]:.3e}')

assumed a0  : 4.082000 A
recovered a0: 4.080510 A
true a0     : 4.080000 A
scale factor: 0.999635
eps_iso fit : -3.651e-04


### 4e. Uncertainty with partial indexing

Only a subset of grains is usually indexed. The correction's uncertainty grows with the sampling scatter.

In [13]:
fractions = [0.1, 0.2, 0.3, 0.5, 0.75, 1.0]
ses = []
for frac in fractions:
    n_keep = max(5, int(frac * N))
    idx = RNG.choice(N, n_keep, replace=False)
    r = ms.correct_d0(
        strains=strains_with_d0_error[idx],
        stiffness=ms.get_stiffness('Cu'),
        orientations=orientations[idx],
        volumes=volumes[idx],
    )
    ses.append(r['uncertainty']['hydrostatic_se_MPa'])

fig, ax = plt.subplots()
ax.plot(fractions, ses, 'o-')
ax.set_xlabel('fraction of grains retained')
ax.set_ylabel('hydrostatic SE [MPa]')
ax.set_title('correction uncertainty vs sampling completeness')
fig.tight_layout()
save_fig(fig, 'fig_04e_uncertainty_vs_sampling')
plt.close(fig)

saved: /Users/hsharma/opt/MIDAS/packages/midas_stress/examples/fig_04e_uncertainty_vs_sampling.png


### 4f. Confidence filtering

`min_confidence` excludes low-confidence grains from the volume-average, without removing them from the final answer (they still receive the correction).

In [14]:
# Synthesise per-grain confidence (GrainsSim all =1; inject variation for the demo)
synth_conf = np.clip(0.9 + 0.2 * RNG.normal(size=N), 0, 1)
thresholds = np.linspace(0.0, 0.95, 10)
shifts = []
for tc in thresholds:
    r = ms.compute_stress(
        strain=strains_with_d0_error,
        stiffness=ms.get_stiffness('Cu'),
        orient=orientations,
        volumes=volumes,
        confidences=synth_conf,
        min_confidence=tc,
    )
    shifts.append(r['hydrostatic_shift'] * 1e3)

fig, ax = plt.subplots()
ax.plot(thresholds, shifts, 'o-')
ax.set_xlabel('min_confidence threshold')
ax.set_ylabel('hydrostatic shift [MPa]')
ax.set_title('stability of the correction across confidence cuts')
fig.tight_layout()
save_fig(fig, 'fig_04f_confidence_filter')
plt.close(fig)

saved: /Users/hsharma/opt/MIDAS/packages/midas_stress/examples/fig_04f_confidence_filter.png


### 4g. What if you don't know the stiffness?

Many materials (battery electrolytes like LLZO, obscure intermetallics, newly synthesised phases) do not have well-characterised single-crystal elastic constants. **The d₀ recovery does not actually require this information.**

**Scale-invariance theorem.** For a free-standing sample, scaling the stiffness by any factor α leaves ε_iso unchanged:

$$\varepsilon_{\text{iso}}' = \frac{(\alpha\mathbf{a})^\top(\alpha\mathbf{b})}{(\alpha\mathbf{a})^\top(\alpha\mathbf{a})} = \frac{\mathbf{a}^\top\mathbf{b}}{\mathbf{a}^\top\mathbf{a}} = \varepsilon_{\text{iso}}$$

because both the response vector $\mathbf{a} = \langle\mathbf{C}_{\text{lab}}\rangle\{\mathbf{I}\}$ and the residual $\mathbf{b} = \langle\sigma\rangle_V$ scale together.

**Cubic case is even cleaner.** For cubic, $\mathbf{C}\{\mathbf{I}\} = 3K\{\mathbf{I}\}$ is purely hydrostatic and rotation-invariant, so the equation collapses to

$$\varepsilon_{\text{iso}} = \tfrac{1}{3}\mathrm{tr}\langle\boldsymbol{\varepsilon}\rangle_V$$

— no stiffness needed, not even a guess for $K$. `ms.recover_d0_cubic_free_standing()` is a convenience wrapper that exposes this directly.

In [15]:
# --- Cubic free-standing: no stiffness required ---
# Make a synthetic LLZO-like sample (cubic, a0 ≈ 12.97 Å for Ia-3d garnet).
a0_llzo = 12.970
lat_llzo = np.tile([a0_llzo]*3 + [90.0]*3, (200, 1))
# Inject 200 ppm per-grain scatter (simulates fit noise)
lat_llzo[:, :3] *= (1 + 200e-6 * RNG.normal(size=(200, 3)))
# Pretend user assumed a wrong reference
a0_assumed = a0_llzo * (1 + 400e-6)
ref_assumed = np.array([a0_assumed]*3 + [90.0]*3)

# Stiffness-free recovery — no C needed!
r = ms.recover_d0_cubic_free_standing(lat_llzo, ref_assumed)
print(f'assumed   a0 = {r["reference_assumed"][0]:.6f} Å')
print(f'recovered a0 = {r["reference_recovered"][0]:.6f} Å  (no stiffness argument)')
print(f'true      a0 = {a0_llzo:.6f} Å')
print(f'eps_iso fit  = {r["eps_iso"]:.3e}')

assumed   a0 = 12.975188 Å
recovered a0 = 12.970063 Å  (no stiffness argument)
true      a0 = 12.970000 Å
eps_iso fit  = -3.952e-04


#### Numerical demonstration of scale invariance

With the full `ms.recover_d0`, pass wildly different stiffness values — the answer should not move.

In [16]:
# Use real orientations from GrainsSim for a clean test
ref_au = np.array([4.080]*3 + [90.0]*3)
ref_wrong_au = ref_au * np.array([1 + 400e-6]*3 + [1]*3)
ref_wrong_au[3:] = 90.0  # keep angles fixed

print(f'{"stiffness (GPa)":<25} {"recovered a0 (Å)":<20} {"eps_iso":<15}')
print('-' * 62)
for label, C in [
    ('Au (C11=193)',            ms.get_stiffness('Au')),
    ('Au x 0.01',               0.01 * ms.get_stiffness('Au')),
    ('Au x 100',                100.0 * ms.get_stiffness('Au')),
    ('W  (C11=522, 3x harder)', ms.get_stiffness('W')),
    ('Al (C11=108)',            ms.get_stiffness('Al')),
    ('random cubic (C11=1)',    ms.cubic_stiffness(C11=1, C12=0, C44=0)),
]:
    r = ms.recover_d0(
        lattice_params=grains['lattice_params'],
        assumed_reference=ref_wrong_au,
        stiffness=C,
        orientations=grains['orientations'],
        volumes=volumes,
    )
    print(f'{label:<25} {r["reference_recovered"][0]:<20.8f} {r["eps_iso"]:<15.6e}')

print()
print('All stiffness magnitudes (varying by 4 orders of magnitude) give the')
print('SAME recovered a0 to machine precision — this is the scale-invariance')
print('theorem in action.')

stiffness (GPa)           recovered a0 (Å)     eps_iso        
--------------------------------------------------------------
Au (C11=193)              4.07998676           -4.032472e-04  
Au x 0.01                 4.07998676           -4.032472e-04  
Au x 100                  4.07998676           -4.032472e-04  
W  (C11=522, 3x harder)   4.07998676           -4.032472e-04  
Al (C11=108)              4.07998676           -4.032472e-04  
random cubic (C11=1)      4.07998676           -4.032472e-04  

All stiffness magnitudes (varying by 4 orders of magnitude) give the
SAME recovered a0 to machine precision — this is the scale-invariance
theorem in action.


#### Non-cubic: anisotropy ratio matters, magnitude does not

For HCP / orthorhombic / lower symmetries, `C{I}` has a deviatoric component whose relative magnitude (e.g. C₃₃/C₁₁ for HCP) enters the response vector. The *magnitude* still cancels, but the *anisotropy* matters. A literature estimate within ±30% is more than sufficient — this is far easier than knowing the exact elastic constants of an obscure material.

In [17]:
# Synthesize a small HCP (Ti-like) free-standing dataset
from midas_stress import get_stiffness, hexagonal_stiffness

# True Ti stiffness values
C11, C12, C13, C33, C44 = 162.4, 92.0, 69.0, 180.7, 46.7
C_true = hexagonal_stiffness(C11, C12, C13, C33, C44)

# Build a dataset with known d0 error
N_ti = 100
orient_ti = random_rotations(N_ti, seed=42)
vol_ti = np.ones(N_ti)
ref_ti_true = np.array([2.951, 2.951, 4.684, 90.0, 90.0, 120.0])
lat_ti = np.tile(ref_ti_true, (N_ti, 1))
ref_ti_assumed = ref_ti_true.copy()
ref_ti_assumed[:3] *= 1.0005  # 500 ppm error

print(f'{"stiffness perturbation":<30} {"recovered a (Å)":<18} {"recovered c (Å)":<18}')
print('-' * 70)
# Correct stiffness
r = ms.recover_d0(lat_ti, ref_ti_assumed, C_true, orient_ti, vol_ti)
print(f'{"exact literature values":<30} {r["reference_recovered"][0]:<18.6f} {r["reference_recovered"][2]:<18.6f}')

# Scale magnitude by 10x (absurd)
r = ms.recover_d0(lat_ti, ref_ti_assumed, 10 * C_true, orient_ti, vol_ti)
print(f'{"10x magnitude (absurd)":<30} {r["reference_recovered"][0]:<18.6f} {r["reference_recovered"][2]:<18.6f}')

# Scale by 0.1x (also absurd)
r = ms.recover_d0(lat_ti, ref_ti_assumed, 0.1 * C_true, orient_ti, vol_ti)
print(f'{"0.1x magnitude (absurd)":<30} {r["reference_recovered"][0]:<18.6f} {r["reference_recovered"][2]:<18.6f}')

# Perturb anisotropy (±30% on C33/C11 ratio)
C_weak_c = hexagonal_stiffness(C11, C12, C13, 0.7 * C33, C44)
C_strong_c = hexagonal_stiffness(C11, C12, C13, 1.3 * C33, C44)
r = ms.recover_d0(lat_ti, ref_ti_assumed, C_weak_c, orient_ti, vol_ti)
print(f'{"-30% anisotropy (C33)":<30} {r["reference_recovered"][0]:<18.6f} {r["reference_recovered"][2]:<18.6f}')
r = ms.recover_d0(lat_ti, ref_ti_assumed, C_strong_c, orient_ti, vol_ti)
print(f'{"+30% anisotropy (C33)":<30} {r["reference_recovered"][0]:<18.6f} {r["reference_recovered"][2]:<18.6f}')

print()
print(f'True values: a = {ref_ti_true[0]}, c = {ref_ti_true[2]}')
print('Even a ±30% anisotropy error shifts the recovery by <1 ppm.')
print('The method is remarkably forgiving of stiffness uncertainty.')

stiffness perturbation         recovered a (Å)    recovered c (Å)   
----------------------------------------------------------------------
exact literature values        2.951001           4.684001          
10x magnitude (absurd)         2.951001           4.684001          
0.1x magnitude (absurd)        2.951001           4.684001          
-30% anisotropy (C33)          2.951001           4.684001          
+30% anisotropy (C33)          2.951001           4.684001          

True values: a = 2.951, c = 4.684
Even a ±30% anisotropy error shifts the recovery by <1 ppm.
The method is remarkably forgiving of stiffness uncertainty.


**Practical guidance**:

- Cubic + free-standing ($σ_\text{app} = 0$): use `ms.recover_d0_cubic_free_standing()`. No stiffness needed.
- Any symmetry + free-standing: use `ms.recover_d0()` with any literature-plausible stiffness. The magnitude cancels; rough anisotropy ratios are sufficient.
- Applied load: stiffness magnitude enters via $\mathbf{a} \cdot σ_\text{app}/(\mathbf{a} \cdot \mathbf{a})$. For typical load levels (100 MPa) and stiffness uncertainties (±20%), the residual bias is ~50 ppm — still usually an order of magnitude smaller than the d₀ error being fixed.

## 5. Stress resolved along directions, planes, and frames

### 5a. Normal stress along a lab direction

In [18]:
def normal_stress(sig, n):
    n = np.asarray(n, dtype=float)
    n = n / np.linalg.norm(n)
    return np.einsum('gij,i,j->g', sig, n, n)

sigma_z = normal_stress(stress_corr, [0, 0, 1])
sigma_x = normal_stress(stress_corr, [1, 0, 0])

fig, ax = plt.subplots()
ax.hist(sigma_z * 1e3, bins=30, alpha=0.6, label='$\\sigma_n$ along z')
ax.hist(sigma_x * 1e3, bins=30, alpha=0.6, label='$\\sigma_n$ along x')
ax.set_xlabel('normal stress [MPa]')
ax.legend()
ax.set_title('per-grain normal stress along two lab-frame directions')
fig.tight_layout()
save_fig(fig, 'fig_05a_normal_stress')
plt.close(fig)

saved: /Users/hsharma/opt/MIDAS/packages/midas_stress/examples/fig_05a_normal_stress.png


### 5b. Shear stress on a plane (traction vector decomposition)

In [19]:
def shear_on_plane(sig, n):
    n = np.asarray(n, dtype=float)
    n = n / np.linalg.norm(n)
    traction = np.einsum('gij,j->gi', sig, n)
    tn = np.einsum('gi,i->g', traction, n)
    normal_part = tn[:, None] * n[None, :]
    shear_vec = traction - normal_part
    return np.linalg.norm(shear_vec, axis=1)

tau_basal = shear_on_plane(stress_corr, [0, 0, 1])
print(f'shear on z-normal plane: mean {tau_basal.mean()*1e3:.2f} MPa, '
      f'max {tau_basal.max()*1e3:.2f} MPa')

shear on z-normal plane: mean 33.76 MPa, max 80.10 MPa


### 5c. Frame rotations (MIDAS ↔ APS, lab ↔ sample)

In [20]:
# Convert orientations + strains into the APS sample frame
sam = ms.grains_midas_to_sample(
    orientations=orientations,
    positions=positions,
    strains=strains,
    omega_deg=0.0,
    target_frame='aps',
)
print('MIDAS mean strain (diag):  ', np.diag(strains.mean(axis=0)))
print('APS  mean strain (diag):  ', np.diag(sam["strains"].mean(axis=0)))
print('(axes are cyclically permuted: MIDAS (x=beam, y=OB, z=up) ↔ APS (x=OB, y=up, z=beam))')

MIDAS mean strain (diag):   [-0.00032191 -0.00029892  0.00099564]
APS  mean strain (diag):   [-0.00029892  0.00099564 -0.00032191]
(axes are cyclically permuted: MIDAS (x=beam, y=OB, z=up) ↔ APS (x=OB, y=up, z=beam))


### 5d. Principal stresses and directions

In [21]:
eigvals, eigvecs = np.linalg.eigh(stress_corr)      # ascending order
sigma_1 = eigvals[:, 2] * 1e3  # MPa
sigma_3 = eigvals[:, 0] * 1e3

fig, ax = plt.subplots()
ax.hist(sigma_1, bins=30, alpha=0.6, label='$\\sigma_1$ (max)', color='C3')
ax.hist(sigma_3, bins=30, alpha=0.6, label='$\\sigma_3$ (min)', color='C0')
ax.set_xlabel('principal stress [MPa]')
ax.legend()
ax.set_title('principal-stress distribution across all grains')
fig.tight_layout()
save_fig(fig, 'fig_05d_principal_stresses')
plt.close(fig)

saved: /Users/hsharma/opt/MIDAS/packages/midas_stress/examples/fig_05d_principal_stresses.png


### 5e. Hydrostatic / deviatoric / von Mises recap

In [22]:
p = ms.hydrostatic(stress_corr)
s = ms.deviatoric(stress_corr)
vm = ms.von_mises(stress_corr)

print(f'hydrostatic mean : {p.mean()*1e3:.2f} MPa (== tr(σ)/3)')
print(f'deviatoric norm  : {np.linalg.norm(s, axis=(1,2)).mean()*1e3:.2f} MPa')
print(f'von Mises mean   : {vm.mean()*1e3:.2f} MPa')

hydrostatic mean : -0.00 MPa (== tr(σ)/3)
deviatoric norm  : 73.21 MPa
von Mises mean   : 89.66 MPa


## 6. CRSS / Schmid / resolved shear — slip-system analysis

Publication-ready slip-system analysis is now first-class in the package. Databases ship for FCC, BCC (including {110}, {112}, {123}), and HCP (basal, prismatic, pyramidal-<a>, pyramidal-<c+a>).

### 6a. Slip-system databases

In [23]:
n_fcc, b_fcc = ms.get_slip_systems('fcc')
print(f'FCC {{111}}<110>      : {n_fcc.shape[0]} systems')

n_bcc, b_bcc = ms.get_slip_systems('bcc')  # {110} + {112}
print(f'BCC {{110}}∪{{112}}<111>: {n_bcc.shape[0]} systems')

n_hcp, b_hcp = ms.get_slip_systems('hcp', c_over_a=ms.HCP_RATIOS['Ti'])
print(f'HCP combined (Ti)    : {n_hcp.shape[0]} systems')
print()
print('Material dispatcher picks sensible defaults:')
for mat in ['Cu', 'Fe', 'Ti']:
    n_mat, _ = ms.get_slip_systems_for_material(mat)
    print(f'  {mat}: {n_mat.shape[0]} systems')

FCC {111}<110>      : 12 systems
BCC {110}∪{112}<111>: 24 systems
HCP combined (Ti)    : 24 systems

Material dispatcher picks sensible defaults:
  Cu: 12 systems
  Fe: 24 systems
  Ti: 24 systems


### 6b. Schmid factor (uniaxial shortcut)

For a uniaxial load along `ell`, the Schmid factor per grain per system is `m = (n_lab · ell)(b_lab · ell)`.

In [24]:
n_sys, b_sys = ms.get_slip_systems('fcc')
m = ms.schmid_factor(orientations, [0, 0, 1], n_sys, b_sys)  # (N, 12)
print(f'Schmid-factor matrix: {m.shape}')
print(f'Max Schmid factor per grain: mean={m.max(axis=1).mean():.3f}, '
      f'single-system ceiling = 0.500 (n, b both at 45° to load)')

fig, ax = plt.subplots()
ax.hist(m.max(axis=1), bins=30, color='steelblue', edgecolor='k')
ax.axvline(0.5, color='k', ls='--', label='single-system ceiling (0.5)')
ax.set_xlabel('max Schmid factor per grain')
ax.legend()
ax.set_title('polycrystal Schmid-factor distribution (FCC, load = z)')
fig.tight_layout()
save_fig(fig, 'fig_06b_schmid_factor')
plt.close(fig)

Schmid-factor matrix: (250, 12)
Max Schmid factor per grain: mean=0.453, single-system ceiling = 0.500 (n, b both at 45° to load)
saved: /Users/hsharma/opt/MIDAS/packages/midas_stress/examples/fig_06b_schmid_factor.png


### 6c. Resolved shear stress from the full stress tensor

Works for any loading mode (uniaxial, biaxial, shear, or the per-grain stress tensor from the equilibrium correction). This is what you want when stresses are heterogeneous across grains.

In [25]:
tau = ms.resolved_shear_stress(stress_corr, orientations, n_sys, b_sys)  # (N, 12)
print(f'|τ| max per grain: mean={np.abs(tau).max(axis=1).mean()*1e3:.1f} MPa')
print(f'|τ| max overall : {np.abs(tau).max()*1e3:.1f} MPa')

# System-by-system activity
fig, ax = plt.subplots()
ax.boxplot(np.abs(tau)*1e3, showfliers=False)
ax.set_xlabel('slip system index')
ax.set_ylabel(r'$|\tau|$ [MPa]')
ax.set_title('resolved shear per system across all grains')
fig.tight_layout()
save_fig(fig, 'fig_06c_resolved_shear')
plt.close(fig)

|τ| max per grain: mean=41.0 MPa
|τ| max overall : 72.4 MPa
saved: /Users/hsharma/opt/MIDAS/packages/midas_stress/examples/fig_06c_resolved_shear.png


### 6d. Dominant / top-K active systems per grain

In [26]:
dom = ms.dominant_slip_system(
    orient=orientations,
    n_crystal=n_sys, b_crystal=b_sys,
    stress=stress_corr,
    top_k=3,
)
# Count how often each system is ranked #1
counts = np.bincount(dom['best_index'], minlength=n_sys.shape[0])

fig, ax = plt.subplots()
ax.bar(range(n_sys.shape[0]), counts, color='teal')
ax.set_xlabel('slip system index')
ax.set_ylabel('grains where it dominates')
ax.set_title('which FCC system dominates, across the polycrystal')
fig.tight_layout()
save_fig(fig, 'fig_06d_dominant_system')
plt.close(fig)

saved: /Users/hsharma/opt/MIDAS/packages/midas_stress/examples/fig_06d_dominant_system.png


### 6e. CRSS classification

Given a critical resolved shear stress (CRSS), classify system activity per grain and report:
- which grains have any active system ("yielded")
- which systems activate most frequently
- the fraction of grains in the plastic regime

In [27]:
# Example CRSS for Cu ~ 25 MPa = 0.025 GPa
crss = 0.025  # GPa, matches stress_corr units

act = ms.active_systems_from_crss(stress_corr, orientations, n_sys, b_sys, crss=crss)

print(f'fraction of grains yielding : {act["fraction_grains_yielding"]*100:.1f} %')
print(f'mean active systems per grain: {act["n_active_per_grain"].mean():.2f}')
print(f'system-activation frequency : {np.round(act["fraction_active_per_system"]*100, 1)} %')

fraction of grains yielding : 92.0 %
mean active systems per grain: 3.65
system-activation frequency : [30.4 34.8 30.8 28.8 31.2 28.4 28.4 31.6 35.2 28.8 26.4 30.4] %


### 6f. Yield-proximity ranking

Rank grains by `max_s |τ_s| / CRSS_s`. A grain is yielding when the score is ≥ 1. Useful for identifying which grains hit plasticity first.

In [28]:
prox = ms.yield_proximity(stress_corr, orientations, n_sys, b_sys, crss=crss)

fig, ax = plt.subplots()
ax.hist(prox['proximity'], bins=40, color='firebrick', edgecolor='k')
ax.axvline(1.0, color='k', ls='--', label='yield threshold')
ax.set_xlabel(r'$\max_s |\tau_s|/\tau_{c}$')
ax.set_ylabel('grain count')
ax.set_title(f'yield proximity (top-{5} grains most stressed)')
ax.legend()
fig.tight_layout()
save_fig(fig, 'fig_06f_yield_proximity')
plt.close(fig)

top_grains = prox['grains_sorted'][:5]
for rank, g in enumerate(top_grains, 1):
    print(f'rank {rank}: grain {g:4d}  proximity {prox["proximity"][g]:.3f}  '
          f'critical system {prox["critical_system"][g]}')

saved: /Users/hsharma/opt/MIDAS/packages/midas_stress/examples/fig_06f_yield_proximity.png
rank 1: grain   23  proximity 2.896  critical system 2
rank 2: grain  116  proximity 2.861  critical system 2
rank 3: grain  245  proximity 2.813  critical system 10
rank 4: grain   44  proximity 2.755  critical system 6
rank 5: grain  200  proximity 2.738  critical system 10


### 6g. Taylor factor (polycrystal yield estimate)

Single-slip upper bound on the polycrystal Taylor factor.

In [29]:
taylor = ms.taylor_factor(orientations, [0, 0, 1], n_sys, b_sys, volumes=volumes)
print(f'single-slip Taylor factor (polycrystal)        : {taylor["M_poly"]:.3f}')
print(f'per-grain M, mean                              : {np.nanmean(taylor["M_per_grain"]):.3f}')
print(f'estimated yield stress = M_poly × CRSS          : {taylor["M_poly"] * crss * 1e3:.1f} MPa')

single-slip Taylor factor (polycrystal)        : 2.225
per-grain M, mean                              : 2.225
estimated yield stress = M_poly × CRSS          : 55.6 MPa


### 6h. HCP slip (Ti example)

For non-cubic materials the per-family CRSS values differ substantially. Here's the usual Ti picture: basal slip has the lowest CRSS, pyramidal the highest.

In [30]:
# Randomly orient some Ti grains
U_ti = random_rotations(200, seed=5)
vol_ti = np.ones(200)

# Apply a uniaxial 300 MPa along z
stress_ti = np.broadcast_to(np.diag([0, 0, 0.300]), (200, 3, 3)).copy()

families = ['hcp_basal', 'hcp_prismatic', 'hcp_pyramidal_a', 'hcp_pyramidal_ca']
crss_ti  = [0.025, 0.060, 0.080, 0.180]  # rough Ti values in GPa

print(f'{"family":<22} {"N sys":>5}   fraction of grains active')
for fam, c in zip(families, crss_ti):
    n_f, b_f = ms.get_slip_systems(fam, c_over_a=ms.HCP_RATIOS['Ti'])
    act_f = ms.active_systems_from_crss(stress_ti, U_ti, n_f, b_f, crss=c)
    print(f'{fam:<22} {n_f.shape[0]:>5}   {act_f["fraction_grains_yielding"]*100:>6.1f} %')

family                 N sys   fraction of grains active
hcp_basal                  3     88.0 %
hcp_prismatic              3     80.0 %
hcp_pyramidal_a            6     72.5 %
hcp_pyramidal_ca          12      0.0 %


## 7. Real MIDAS output walk-through

Connecting everything: read a MIDAS CSV, convert to APS sample frame, run the full pipeline.

In [31]:
# For this demo we re-use our synthetic strain (GrainsSim has zero strain);
# replace with `grains["strain"]` for a file that actually has values.

sam = ms.grains_midas_to_sample(
    orientations=grains['orientations'],
    positions=grains['positions'],
    strains=strains,        # or grains['strain'] when you have real data
    omega_deg=0.0,
    target_frame='aps',
)

result = ms.compute_stress(
    strain=sam['strains'],
    stiffness=ms.get_stiffness('Cu'),
    orient=sam['orientations'],
    volumes=(4/3)*np.pi*grains['radii']**3,
    confidences=grains['confidences'],
    min_confidence=0.5,
)

print(f'polycrystal mean von Mises : {result["von_mises"].mean()*1e3:.1f} MPa')
print(f'hydrostatic correction     : {result["hydrostatic_shift"]*1e3:.2f} MPa')
print(f'uncertainty (hydro SE)     : {result["uncertainty"]["hydrostatic_se_MPa"]:.2f} MPa')

# Spatial map of von Mises
pos = sam['positions']
fig, ax = plt.subplots()
sc = ax.scatter(pos[:, 0], pos[:, 1], c=result['von_mises']*1e3,
                s=grains['radii']/5, cmap='viridis')
ax.set_xlabel('x [µm]'); ax.set_ylabel('y [µm]')
ax.set_aspect('equal')
plt.colorbar(sc, ax=ax, label='von Mises [MPa]')
ax.set_title(f'{N} grains from {os.path.basename(GRAINS_CSV)}')
fig.tight_layout()
save_fig(fig, 'fig_07_spatial_map')
plt.close(fig)

polycrystal mean von Mises : 89.7 MPa
hydrostatic correction     : -51.38 MPa
uncertainty (hydro SE)     : 0.00 MPa
saved: /Users/hsharma/opt/MIDAS/packages/midas_stress/examples/fig_07_spatial_map.png


## 8. Reference cheat-sheet

### Units

| Quantity         | Unit                             |
|------------------|----------------------------------|
| Stiffness `C`    | GPa (library default)            |
| Strain           | dimensionless (order 1e-3)       |
| Stress           | same as stiffness (GPa → x1000 for MPa) |
| Lattice `a,b,c`  | Ångström                        |
| Lattice angles   | degrees                          |
| Euler angles     | radians (MIDAS convention)       |
| Positions        | micrometre                       |

### Voigt–Mandel

```
v = [T_xx, T_yy, T_zz, sqrt(2)*T_xy, sqrt(2)*T_xz, sqrt(2)*T_yz]
```
Preserves `||T||_F == ||v||_2`. The sqrt(2) makes matrix-vector products give the correct tensor rotations.

### Frames

```
MIDAS (ESRF) :  x=beam   y=OB   z=up
APS (Park)   :  x=OB     y=up   z=beam
```

The two are related by a cyclic axis permutation (`R_MIDAS_TO_APS`).

### Strain convention

`grains['strain']` = d-spacing (strain-gauge) form. Fit directly from per-reflection Δd/d₀, so it is tied to the raw observable (peak positions). Historically called the 'Kenesei' form in MIDAS.

`grains['strain_lattice']` = lattice-parameter form, ε = ½(F+Fᵀ)−I with F = A·A₀⁻¹. Mathematically equivalent to the d-spacing form at HEDM strain magnitudes; use for comparison only. Historically called the 'Fable-Beaudoin' form in MIDAS.

## 9. Decision tree — which function?

```
Do you have per-grain strain?
  |
  +-- No, just orientations + applied uniaxial load
  |   → ms.schmid_factor(orient, load, n, b)
  |
  +-- Yes
      |
      +-- Is d0 known to high precision?
      |   Yes → ms.hooke_stress(...) (no correction)
      |   No  → ms.correct_d0(...) (two-step)
      |          or ms.recover_d0(...) (from lattice params)
      |
      +-- Is applied macroscopic stress known?
          Yes → pass it to compute_stress / correct_d0 via applied_stress=
          No  → omit; assumes free-standing (zero applied)
      |
      +-- Want slip/CRSS analysis on the per-grain stress?
          → ms.resolved_shear_stress(stress, orient, n, b)
          → ms.active_systems_from_crss(stress, orient, n, b, crss)
          → ms.yield_proximity(...)
          → ms.taylor_factor(...)
```

## 10. What this notebook does NOT cover

- Fitting single-crystal elastic constants from experimental data.
- Plastic deformation / crystal plasticity FEM coupling.
- Peak-fitting / strain extraction from raw diffraction patterns (upstream of this package — use MIDAS, hexrd, ImageD11, or DAXM).
- Fitting CRSS values from a stress-strain curve (only resolved-shear computation is covered; fitting requires external yield data).
- GPU / PyTorch differentiable path (`midas_stress.torch_backend`).

For any of the above, reach out or extend the package — all functions here are plain NumPy, so dropping in new analysis is straightforward.